# Stereo & Oblique Camera Flight Planning

This notebook demonstrates HyPlan's support for:
- **Tilted (oblique) cameras** — footprint and GSD variation with off-nadir viewing
- **Stereo photogrammetry** — base-height ratio and vertical accuracy from overlap
- **Multi-camera rigs** — the `MultiCameraRig` class and the QUAKES-I factory

Reference: Donnellan et al. (2025), *Earth and Space Science*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hyplan.units import ureg
from hyplan.frame_camera import FrameCamera, MultiCameraRig

## 1. Nadir vs Tilted Camera Comparison

Compare a standard nadir camera with a 15° forward-tilted version.

In [ ]:
# Common sensor specs
kwargs = dict(
    sensor_width=36 * ureg.mm,
    sensor_height=24 * ureg.mm,
    focal_length=50 * ureg.mm,
    resolution_x=6000,
    resolution_y=4000,
    frame_rate=2 * ureg.Hz,
    f_speed=2.8,
)

nadir = FrameCamera(name="Nadir", **kwargs)
tilted = FrameCamera(name="Tilted 15°", tilt_angle=15.0, tilt_direction=0.0, **kwargs)

alt = 1000 * ureg.meter

print("=== Footprint at 1000 m AGL ===")
for cam in [nadir, tilted]:
    fp = cam.footprint_at(alt)
    print(f"\n{cam.name}:")
    print(f"  Width:  {fp['width'].to('meter'):.1f}")
    print(f"  Height: {fp['height'].to('meter'):.1f}")
    if 'height_near' in fp:
        print(f"  Height near: {fp['height_near'].to('meter'):.1f}")
        print(f"  Height far:  {fp['height_far'].to('meter'):.1f}")

In [ ]:
print("=== Ground Sample Distance at 1000 m AGL ===")
for cam in [nadir, tilted]:
    gsd = cam.ground_sample_distance(alt)
    print(f"\n{cam.name}:")
    print(f"  GSD x (across): {gsd['x'].to('cm'):.2f}")
    print(f"  GSD y (along):  {gsd['y'].to('cm'):.2f}")
    if 'y_near' in gsd:
        print(f"  GSD y near:     {gsd['y_near'].to('cm'):.2f}")
        print(f"  GSD y far:      {gsd['y_far'].to('cm'):.2f}")

## 2. GSD Variation with Tilt Angle

In [ ]:
tilt_angles = np.arange(0, 46, 5)
gsd_center = []

for t in tilt_angles:
    cam = FrameCamera(name=f"T{t}", tilt_angle=float(t), **kwargs)
    gsd = cam.ground_sample_distance(alt)
    gsd_center.append(gsd['y'].to('cm').magnitude)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tilt_angles, gsd_center, 'o-')
ax.set_xlabel('Tilt angle (degrees)')
ax.set_ylabel('Center GSD along-track (cm)')
ax.set_title('GSD Degradation with Camera Tilt (1000 m AGL)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Stereo Photogrammetry

Estimate the base-height ratio and vertical accuracy from forward overlap.

In [ ]:
alt = 2000 * ureg.meter
overlaps = [60, 70, 80, 90]

print(f"{'Overlap':>10} {'B/H':>8} {'σ_z (m)':>10}")
print("-" * 30)
for olap in overlaps:
    bh = nadir.base_height_ratio(alt, overlap_pct=olap)
    vz = nadir.vertical_accuracy(alt, overlap_pct=olap, sigma_parallax=0.5)
    print(f"{olap:>9}% {bh:>8.3f} {vz.to('meter').magnitude:>10.3f}")

## 4. Range Accuracy (Donnellan et al. 2025 formula)

For convergent stereo systems like QUAKES-I, range accuracy is:

$$\sigma_R = \frac{R^2 \cdot \sigma_q}{B}$$

where $R$ is slant range, $\sigma_q$ is angular measurement uncertainty, and $B$ is baseline.

In [ ]:
# QUAKES-I parameters from the paper
alt_gv = 12500 * ureg.meter
sigma_q = 0.023e-3  # 0.023 mrad → radians

baselines = np.arange(1000, 8001, 500) * ureg.meter

quakes_cam = FrameCamera(
    name="QUAKES-I cam",
    sensor_width=35 * ureg.mm,
    sensor_height=24 * ureg.mm,
    focal_length=100 * ureg.mm,
    resolution_x=5120,
    resolution_y=3840,
    frame_rate=2 * ureg.Hz,
    f_speed=2.8,
    tilt_angle=11.3,
)

range_acc = [quakes_cam.range_accuracy(alt_gv, b, sigma_q=sigma_q).to('meter').magnitude
             for b in baselines]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([b.magnitude / 1000 for b in baselines], range_acc, 'o-')
ax.set_xlabel('Baseline (km)')
ax.set_ylabel('Range accuracy σ_R (m)')
ax.set_title('QUAKES-I Stereo Range Accuracy vs Baseline (12.5 km AGL)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Multi-Camera Rig: QUAKES-I

The QUAKES-I system uses 8 cameras: 4 forward-looking and 4 aft-looking, each tilted 11.3° from nadir.

In [ ]:
rig = MultiCameraRig.quakes_i()

print(f"Rig: {rig.name}")
print(f"Number of cameras: {len(rig)}")
print(f"\nCamera labels:")
for entry in rig.cameras:
    cam = entry['camera']
    print(f"  {entry['label']:>8}: tilt={cam.tilt_angle:.1f}° dir={cam.tilt_direction:.0f}°")

In [ ]:
alt_gv = 12500 * ureg.meter

print(f"\n=== QUAKES-I at {alt_gv} ===")
print(f"Swath width: {rig.swath_width(alt_gv).to('km'):.2f}")

gsd = rig.ground_sample_distance(alt_gv)
print(f"Finest GSD x: {gsd['x'].to('cm'):.1f}")
print(f"Finest GSD y: {gsd['y'].to('cm'):.1f}")

print(f"\nStereo pairs: {len(rig.stereo_pairs())}")
for result in rig.composite_base_height_ratio(alt_gv):
    print(f"  {result['pair'][0]} ↔ {result['pair'][1]}: B/H = {result['bh_ratio']:.3f}")

In [ ]:
# Flight line spacing for a survey
for sidelap in [30, 50, 60]:
    sp = rig.line_spacing(alt_gv, sidelap_pct=sidelap)
    print(f"Sidelap {sidelap}%: line spacing = {sp.to('km'):.2f}")

## 6. Simple Forward+Aft Stereo Rig

Build a custom 2-camera rig and evaluate stereo performance.

In [ ]:
# Custom stereo rig
fwd = FrameCamera(
    name="Forward", tilt_angle=20.0, tilt_direction=0.0, **kwargs
)
aft = FrameCamera(
    name="Aft", tilt_angle=20.0, tilt_direction=180.0, **kwargs
)

stereo_rig = MultiCameraRig("Custom Stereo", [
    {"camera": fwd, "label": "forward"},
    {"camera": aft, "label": "aft"},
])

alt = 3000 * ureg.meter
pairs = stereo_rig.stereo_pairs()
bh_results = stereo_rig.composite_base_height_ratio(alt)

print(f"Stereo pairs found: {len(pairs)}")
print(f"Convergent B/H = tan(20°) + tan(20°) = {bh_results[0]['bh_ratio']:.3f}")
print(f"Swath width at {alt}: {stereo_rig.swath_width(alt).to('meter'):.0f}")

# Range accuracy at various baselines
print(f"\nRange accuracy at {alt}:")
for b_km in [1, 2, 3, 5]:
    b = b_km * 1000 * ureg.meter
    ra = fwd.range_accuracy(alt, b)
    print(f"  Baseline {b_km} km: σ_R = {ra.to('meter'):.2f}")

## 7. QUAKES-I Ground Footprint Map

Project each camera's sensor perimeter onto a flat ground plane using
`ground_footprint()`.  The cross-track angular offsets from the
QUAKES-I design (inner pair 17.1°, outer pair 40.5°) spread the four
cameras across a ~60° combined cross-track field of view.

`ground_footprint` supports two modes:
- **Flat-ground** (default): returns Shapely Polygon with local `(x, y, 0)` coordinates — fast, no DEM needed
- **Terrain-aware**: pass `lat`, `lon`, `altitude_msl` to intersect rays with a DEM — returns Shapely Polygon with `(lon, lat, elevation)`

In [ ]:
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

rig = MultiCameraRig.quakes_i()
alt_gv = 12500 * ureg.meter

footprints = rig.ground_footprint(alt_gv)

fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)

for ax, title, fp_slice in zip(
    axes,
    ["Forward array (11.3° fwd)", "Aft array (11.3° aft)"],
    [footprints[:4], footprints[4:]],
):
    patches = []
    colors = []
    cmap = plt.cm.Set2

    for i, fp in enumerate(fp_slice):
        poly = fp["polygon"]
        coords_km = [(x / 1000, y / 1000) for x, y, *_ in poly.exterior.coords[:-1]]
        patches.append(MplPolygon(coords_km, closed=True))
        colors.append(cmap(i / 4))
        cx, cy = poly.centroid.x / 1000, poly.centroid.y / 1000
        ax.text(cx, cy, fp["label"], ha="center", va="center",
                fontsize=8, fontweight="bold")

    pc = PatchCollection(patches, facecolors=colors, edgecolors="k",
                         linewidths=1.2, alpha=0.45)
    ax.add_collection(pc)
    ax.plot(0, 0, "k+", markersize=12, markeredgewidth=2, label="Nadir")
    ax.set_xlabel("Cross-track (km)")
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")

axes[0].set_ylabel("Along-track (km)")
fig.suptitle(f"QUAKES-I Ground Footprints at {alt_gv.to('km'):.1f} AGL",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
from shapely.ops import unary_union
from matplotlib.patches import Patch

# Combined view: all 8 cameras overlaid
fig, ax = plt.subplots(figsize=(12, 10))

fwd_color = "#3182bd"  # blue
aft_color = "#e6550d"  # orange

for i, fp in enumerate(footprints):
    poly = fp["polygon"]
    coords_km = [(x / 1000, y / 1000) for x, y, *_ in poly.exterior.coords[:-1]]
    color = fwd_color if i < 4 else aft_color
    mpl_poly = MplPolygon(coords_km, closed=True, facecolor=color, edgecolor="k",
                          alpha=0.3, linewidth=1.2)
    ax.add_patch(mpl_poly)
    cx, cy = poly.centroid.x / 1000, poly.centroid.y / 1000
    ax.text(cx, cy, fp["label"], ha="center", va="center",
            fontsize=7, fontweight="bold", color="k")

# Mark nadir and flight direction
ax.plot(0, 0, "k+", markersize=14, markeredgewidth=2.5, zorder=5)
ax.annotate("Nadir", (0, 0), textcoords="offset points", xytext=(8, -12),
            fontsize=9, fontstyle="italic")
ax.annotate("", xy=(0, 1.5), xytext=(0, 0.5),
            arrowprops=dict(arrowstyle="->", lw=1.5, color="gray"))
ax.text(0.15, 1.0, "Flight\ndirection", fontsize=8, color="gray")

ax.legend(handles=[
    Patch(facecolor=fwd_color, alpha=0.4, edgecolor="k", label="Forward array"),
    Patch(facecolor=aft_color, alpha=0.4, edgecolor="k", label="Aft array"),
], loc="upper left", fontsize=10)

ax.set_xlabel("Cross-track (km)")
ax.set_ylabel("Along-track (km)")
ax.set_title(f"QUAKES-I Combined Footprints — {alt_gv.to('km'):.1f} AGL",
             fontsize=13, fontweight="bold")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print total cross-track coverage using Shapely
all_poly = unary_union([fp["polygon"] for fp in footprints])
bounds = all_poly.bounds  # (minx, miny, maxx, maxy)
total_cross = (bounds[2] - bounds[0]) / 1000

fwd_poly = unary_union([fp["polygon"] for fp in footprints[:4]])
aft_poly = unary_union([fp["polygon"] for fp in footprints[4:]])
print(f"Total cross-track coverage: {total_cross:.1f} km")
print(f"Forward array extends to:   +{fwd_poly.bounds[3]/1000:.1f} km along-track")
print(f"Aft array extends to:       {aft_poly.bounds[1]/1000:.1f} km along-track")

## 8. Terrain-Aware Footprint

When a camera position and DEM are available, `ground_footprint`
intersects sensor edge rays with the actual terrain surface instead of
assuming flat ground. Pass `lat`, `lon`, and `altitude_msl` to enable
terrain mode. The Copernicus GLO-30 DEM is auto-downloaded if no
`dem_file` is specified.

The returned Shapely Polygon has 3D coordinates `(lon, lat, elevation)`.

In [ ]:
# Single camera: terrain-aware footprint over San Bernardino mountains
cam = FrameCamera(
    name="Terrain test",
    sensor_width=35 * ureg.mm,
    sensor_height=24 * ureg.mm,
    focal_length=100 * ureg.mm,
    resolution_x=5120,
    resolution_y=3840,
    frame_rate=2 * ureg.Hz,
    f_speed=2.8,
    tilt_angle=11.3,
)

alt = 12500 * ureg.meter
poly = cam.ground_footprint(
    alt,
    lat=34.05, lon=-117.25, altitude_msl=6000.0,
    heading=270.0,
)

coords = list(poly.exterior.coords)
print(f"Polygon vertices: {len(coords) - 1}")
print(f"Area: {poly.area:.6f} deg^2")
print("\nFirst 8 vertices (lon, lat, elev):")
for i, (lon, lat, elev) in enumerate(coords[:8]):
    print(f"  {i}: lat={lat:.6f}, lon={lon:.6f}, elev={elev:.1f} m")

In [ ]:
# Full QUAKES-I rig: terrain-aware footprints
rig = MultiCameraRig.quakes_i()

fps = rig.ground_footprint(
    alt,
    lat=34.05, lon=-117.25, altitude_msl=13000.0,
    heading=270.0,
)

print(f"Number of footprints: {len(fps)}")
print(f"Vertices per polygon: {len(fps[0]['polygon'].exterior.coords) - 1}\n")

for fp in fps:
    poly = fp["polygon"]
    coords = np.array(poly.exterior.coords)
    mean_elev = coords[:, 2].mean()
    print(f"  {fp['label']:>8}: mean ground elev = {mean_elev:.0f} m, "
          f"lat range [{coords[:,1].min():.4f}, {coords[:,1].max():.4f}]")

In [ ]:
import folium

# Map the terrain-aware footprints
all_coords = []
for fp in fps:
    all_coords.extend(fp["polygon"].exterior.coords)
all_coords = np.array(all_coords)
center_lat = all_coords[:, 1].mean()
center_lon = all_coords[:, 0].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=9,
               tiles="CartoDB positron")

# Aircraft position marker
folium.Marker(
    [34.05, -117.25],
    popup="Aircraft (13 km MSL)",
    icon=folium.Icon(color="black", icon="plane", prefix="fa"),
).add_to(m)

fwd_color = "#3182bd"
aft_color = "#e6550d"

for i, fp in enumerate(fps):
    poly = fp["polygon"]
    color = fwd_color if i < 4 else aft_color
    coords_3d = list(poly.exterior.coords)
    mean_elev = np.mean([c[2] for c in coords_3d])

    folium.GeoJson(
        poly.__geo_interface__,
        style_function=lambda x, c=color: {
            "fillColor": c, "color": c,
            "weight": 2, "fillOpacity": 0.25,
        },
        tooltip=f"{fp['label']} (mean elev {mean_elev:.0f} m)",
    ).add_to(m)

m